In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sharankumare/mcq-test/test.csv
/kaggle/input/datasets/sharankumare/mcq-train/train.csv


In [2]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/datasets/sharankumare/mcq-train/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.2 MB/s eta 0:00:00
Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
result = zs(prompt_150 , candidate_labels = labels_150)

for label , score in zip(result['labels'] , result['scores']):
    print(label , round(score,3))

correct_score = result['scores'][result['labels'].index(ans_150)]

print("\nCorrect Answer:", ans_150)
print("Probability:", round(correct_score,3))

The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." 0.384
The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." 0.379
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity." 0.093
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos." 0.079
The butterfly effect is the phenomenon t

In [5]:
# Row 150 prompt
query = train.iloc[150]["prompt"]

# Embed the prompt
query_embedding = model.encode([query])

# Retrieve Top-10
distances, indices = index.search(query_embedding, k=10)

print("Retrieved indices:", indices[0])

# Find the rank of the correct document (KB index = 150)
if 150 in indices[0]:
    rank = list(indices[0]).index(150) + 1
    print("Rank of correct document:", rank)
else:
    print("Correct document not found in Top-10")

Retrieved indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of correct document: 10


In [6]:
from sentence_transformers import CrossEncoder
import numpy as np

# Load Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Top-10 document indices from FAISS
retrieved_indices = indices[0]

# Corresponding documents
docs_10 = [kb[i] for i in retrieved_indices]

# Create (question, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Cross-Encoder scores
ce_scores = cross_encoder.predict(pairs)

# Sort by score (highest first)
sorted_idx = np.argsort(ce_scores)[::-1]

# Re-ranked document indices
reranked_indices = retrieved_indices[sorted_idx]

print("Re-ranked document indices:")
print(reranked_indices)

# Find rank of the correct document (KB index 150)
if 150 in reranked_indices:
    rank = list(reranked_indices).index(150) + 1
    print("Rank of correct document:", rank)
else:
    print("Correct document not found in Top-10")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranked document indices:
[ 150 1906  847 1693 1269 1532  168  576 1701  663]
Rank of correct document: 1


In [7]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Prompt from row 42
prompt_42 = train.iloc[42]["prompt"]

# Embed the prompt
query_embedding = model.encode([prompt_42])

# Retrieve Top-5 documents
_, indices = index.search(query_embedding, k=5)

# Concatenate documents
retrieved_docs = [kb[i] for i in indices[0]]
concatenated_docs = " ".join(retrieved_docs)

# Create final input string
text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Tokenize (NO truncation)
tokens = tokenizer(text, truncation=False)

# Total number of tokens
print("Total tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


In [8]:
# Retrieve the TRUE document from KB
true_document = kb[150]

# Create RAG prompt
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

# Zero-shot classification
result = zs(rag_prompt, candidate_labels=labels_150)

# Probability of the correct answer
correct_score = result["scores"][result["labels"].index(ans_150)]

print("Correct Answer:", ans_150)
print("Probability:", round(correct_score, 3))

Correct Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.989


In [9]:
# Wrong (adversarial) document
wrong_document = kb[999]

# Create adversarial RAG prompt
adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

# Zero-shot classification
result = zs(adversarial_prompt, candidate_labels=labels_150)

# Probability of the correct answer
correct_score = result["scores"][result["labels"].index(ans_150)]

print("Correct Answer:", ans_150)
print("Probability:", round(correct_score, 3))

Correct Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.529


In [10]:
hits = 0

for i in range(100):

    # Query
    prompt = train.iloc[i]["prompt"]

    # Correct answer text
    correct_doc = str(train.iloc[i][train.iloc[i]["answer"]])

    # Embed query
    query_embedding = model.encode([prompt])

    # Retrieve Top-5
    _, indices = index.search(query_embedding, k=5)

    # Retrieved documents
    retrieved_docs = [kb[j] for j in indices[0]]

    # Check if correct answer is present
    if any(correct_doc in doc for doc in retrieved_docs):
        hits += 1

# Hit Rate
hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1))
print(round(hit_rate, 1))

Hits: 73
Hit Rate: 73.0
73.0


In [11]:
import numpy as np

map3_scores = []

for i in range(20):

    # Query
    prompt = str(train.iloc[i]["prompt"])

    # Retrieve
    query_embedding = model.encode([prompt])
    _, indices = index.search(query_embedding, k=5)

    retrieved_docs = [kb[j] for j in indices[0]]

    # Rerank
    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # Augment
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # Candidate Labels
    candidate_labels = [
        str(train.iloc[i]["A"]),
        str(train.iloc[i]["B"]),
        str(train.iloc[i]["C"]),
        str(train.iloc[i]["D"]),
        str(train.iloc[i]["E"])
    ]

    # Predict
    result = zs(rag_prompt, candidate_labels=candidate_labels)

    # Top-3 predicted option texts
    top3_text = result["labels"][:3]

    # Convert texts back to letters
    text_to_letter = {
        str(train.iloc[i]["A"]): "A",
        str(train.iloc[i]["B"]): "B",
        str(train.iloc[i]["C"]): "C",
        str(train.iloc[i]["D"]): "D",
        str(train.iloc[i]["E"]): "E",
    }

    top3_letters = [text_to_letter[t] for t in top3_text]

    # MAP@3 
    true_answer = train.iloc[i]["answer"]

    if true_answer == top3_letters[0]:
        score = 1.0
    elif true_answer == top3_letters[1]:
        score = 1/2
    elif true_answer == top3_letters[2]:
        score = 1/3
    else:
        score = 0.0

    map3_scores.append(score)

# Final MAP@3
final_map3 = np.mean(map3_scores)

print("MAP@3:", round(final_map3, 3))

print(round(final_map3, 3))

MAP@3: 0.975
0.975
